# PD-L1 계정별 Grade PCC · CCC 계산

프로젝트 선택 화면에서 **비교 결과 ZIP**을 다운로드하고 압축을 풀어주세요. 아래 `ACCOUNT_FOLDER`에 **계정별 폴더**, **특정 계정 폴더**, 또는 **압축 해제한 프로젝트 폴더** 경로를 입력한 뒤 위에서부터 실행하면 됩니다.

- 계산 함수는 같은 디렉터리의 `pdl1_pcc.py`에 분리되어 있습니다. 백엔드나 추가 통계 패키지 없이 실행합니다.
- 각 계정의 **전문의 Grade와 해당 측정의 AI Grade(0·1·2 코드)**를 슬라이드별로 대응시켜 PCC를 계산합니다. TPS 기록도 별도로 계산합니다.
- 프로젝트·모델·점수 종류별로 나누며, 계정당 같은 슬라이드의 반복 측정은 **최신 기록 하나**를 사용합니다.
- `비교결과.json`을 우선 사용하며, JSON이 없는 계정 폴더는 `비교결과.csv`를 읽습니다. AI 점수가 포함된 최신 내보내기 형식이 필요합니다.
- 누락·비정상 점수는 대응 쌍 전체를 제외합니다. 슬라이드가 2개 미만이거나 한쪽 Grade가 모두 같으면 PCC는 공란이고 사유가 표시됩니다.

**종합 과정:** 슬라이드별 Grade 과반 다수결 → 다수 Grade에 속한 판독자들의 CPS 중앙값 → 다수결 Grade와 같은 슬라이드 AI Grade를 0·1·2로 대응 → 종합 PCC. 과반이 없거나 1인뿐인 슬라이드는 종합 PCC에서 제외됩니다. 종합 계산에는 전체 판독자의 폴더가 들어 있는 `계정별` 상위 폴더를 지정하세요.


## PCC 공식

$$
r = \frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}
$$

$x_i$는 전문의 Grade 코드, $y_i$는 같은 슬라이드의 AI Grade 코드입니다. **CPS <1 → 0, 1 이상 10 미만 → 1, 10 이상 → 2**입니다. TPS는 1·50 경계로 0·1·2를 부여합니다. 종합 PCC의 $x_i$에는 다수결 Grade 코드를 사용합니다. PCC는 −1~1이며, 선형 상관을 나타냅니다. 점수의 절대적인 일치도를 의미하지는 않습니다.

계산 함수: `pearson_pcc(reference, ai)` → PCC 또는 계산 불가 시 `None`.

### CCC 공식 (Grade 기준)

$$\rho_c = \frac{2\operatorname{cov}(x,y)}{\operatorname{var}(x)+\operatorname{var}(y)+(\bar{x}-\bar{y})^2}$$

PCC와 동일한 **Grade 0·1·2**를 입력합니다. 분산·공분산은 1/N 기준입니다. 종합 CCC는 다수결 Grade 대 AI Grade입니다. CPS 중앙값은 별도 보고값으로 유지합니다. 대응 슬라이드가 2개 미만이거나 분모가 0이면 공란과 사유를 표시합니다.

함수: `lin_ccc(reference_grade, ai_grade)`.


In [ ]:
from pathlib import Path
import sys
import importlib

# 노트북을 저장소 루트 또는 standalone_detection에서 실행해도 함수를 찾습니다.
helper_dir = next(
    (candidate for parent in [Path.cwd(), *Path.cwd().parents]
     for candidate in (parent, parent / "standalone_detection")
     if (candidate / "pdl1_pcc.py").is_file()),
    None,
)
if helper_dir is None:
    raise FileNotFoundError("노트북과 pdl1_pcc.py를 같은 폴더에 두고 실행해주세요.")
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
# 이미 실행 중인 커널에도 디스크의 최신 함수 정의를 반영합니다.
importlib.invalidate_caches()
import pdl1_pcc
importlib.reload(pdl1_pcc)
from pdl1_pcc import pearson_pcc, lin_ccc, calculate_account_pcc, calculate_consensus_pcc, grade_majority_median, score_to_grade_code, save_pcc_csv


In [ ]:
# 이 경로만 실제 압축 해제한 폴더 경로로 바꾸세요.
# 예: Path("/home/user/Downloads/Test_PD-L1_comparison/계정별")
# 특정 계정만: Path("/home/user/Downloads/Test_PD-L1_comparison/계정별/reader_a")
ACCOUNT_FOLDER = Path("../data/계정별")


In [ ]:
results = calculate_account_pcc(ACCOUNT_FOLDER)

from html import escape
from IPython.display import HTML, display


def show_table(rows):
    if not rows:
        print("저장된 비교 기록이 없습니다.")
        return
    columns = list(rows[0])
    def show_value(value):
        if value is None:
            return ""
        return f"{value:.8f}" if isinstance(value, float) else str(value)
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(show_value(row[column]))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("계정별 Grade PCC · CCC")
show_table(results)


## 슬라이드별 합의 과정과 종합 PCC

예: A=1, B=3, C=80이면 `1 ≤ CPS < 10`이 2표로 과반입니다. 해당 Grade인 A·B의 CPS 중앙값 **2**를 합의 점수로 사용하고 C=80은 중앙값 산출에서 제외합니다.

짝수 인원의 중앙값은 가운데 두 값의 평균입니다. Grade 동률·과반 없음·1인 평가에서는 합의 Grade와 CPS 모두 공란입니다. 계정별 AI 점수가 서로 다르면 종합 AI 점수를 선택하지 않고 해당 슬라이드를 제외합니다.

아래 첫 표에 **판독자별 점수, Grade 득표, 중앙값 산출 계정, 합의 중앙값, AI 점수**를 표시합니다. 두 번째 표는 다수결 Grade와 AI Grade 사이의 PCC입니다.


In [ ]:
consensus_slides, consensus_results = calculate_consensus_pcc(ACCOUNT_FOLDER)
print("슬라이드별 Grade 다수결 → 해당 판독자 CPS 중앙값")
show_table(consensus_slides)
print("AI Grade 대 다수결 Grade PCC · CCC")
show_table(consensus_results)


## 결과 CSV 저장 (선택)

아래 셀을 실행하면 입력 폴더에 다음 파일을 저장합니다.

- `계정별_PCC_결과.csv`: 계정별 AI Grade 대 전문의 Grade PCC
- `슬라이드별_합의_산출과정.csv`: Grade 득표, 중앙값에 포함한 계정, 합의 점수·AI 점수 및 양쪽 Grade 코드
- `종합_PCC_결과.csv`: AI Grade 대 다수결 Grade PCC

같은 이름의 기존 결과는 덮어씁니다. 원본 비교 기록은 변경하지 않습니다.

계정별·종합 결과 CSV에는 PCC와 CCC가 모두 포함되며, 양쪽 모두 Grade 기준입니다.


In [ ]:
OUTPUT_CSV = ACCOUNT_FOLDER / "계정별_PCC_결과.csv"
saved_path = save_pcc_csv(results, OUTPUT_CSV)
print(f"저장 완료: {saved_path}")
if consensus_slides:
    print(save_pcc_csv(consensus_slides, ACCOUNT_FOLDER / "슬라이드별_합의_산출과정.csv"))
if consensus_results:
    print(save_pcc_csv(consensus_results, ACCOUNT_FOLDER / "종합_PCC_결과.csv"))


## 점수 배열로 직접 계산 (선택)

폴더 없이도 함수를 호출할 수 있습니다. 두 배열은 **같은 슬라이드 순서**여야 합니다. `score_to_grade_code`로 CPS/TPS를 Grade 코드로 바꾼 뒤 PCC 함수에 전달합니다.
